In [1]:
# =============================================================================
# CELL C1: Upload the 3 Downloaded Model CSVs
# =============================================================================
!pip install -q pandas nltk

from google.colab import files
import pandas as pd
import numpy as np

print("📤 Upload all 3 files now: captions_LLaVA-1.5-7B.csv, "
      "captions_Qwen2.5-VL-3B.csv, captions_Qwen3-VL-4B.csv")
uploaded = files.upload()

model_dfs = []
for filename in uploaded.keys():
    df = pd.read_csv(filename)
    model_dfs.append(df)
    print(f"✅ Loaded {filename}: {len(df)} rows | model = {df['model'].iloc[0]}")

📤 Upload all 3 files now: captions_LLaVA-1.5-7B.csv, captions_Qwen2.5-VL-3B.csv, captions_Qwen3-VL-4B.csv


Saving captions_LLaVA-1.5-7B.csv to captions_LLaVA-1.5-7B.csv
Saving captions_Qwen3-VL-4B .csv to captions_Qwen3-VL-4B .csv
Saving captions_Qwen2.5-VL-3B.csv to captions_Qwen2.5-VL-3B.csv
✅ Loaded captions_LLaVA-1.5-7B.csv: 500 rows | model = LLaVA-1.5-7B
✅ Loaded captions_Qwen3-VL-4B .csv: 500 rows | model = Qwen3-VL-4B
✅ Loaded captions_Qwen2.5-VL-3B.csv: 500 rows | model = Qwen2.5-VL-3B


In [2]:
# =============================================================================
# CELL C2: Verify All 3 Models Used the SAME 500 Real Images
# =============================================================================
image_id_sets = {df["model"].iloc[0]: set(df["image_id"].astype(str)) for df in model_dfs}
ref_set = next(iter(image_id_sets.values()))
all_match = all(s == ref_set for s in image_id_sets.values())

print(f"🔍 Same-500-images check: {'✅ PASSED' if all_match else '❌ FAILED'}")
for name, ids in image_id_sets.items():
    print(f"   {name}: {len(ids)} unique images")

assert all_match, "Models used different images — re-check RANDOM_SEED/Cell 2 consistency across notebooks."

🔍 Same-500-images check: ✅ PASSED
   LLaVA-1.5-7B: 500 unique images
   Qwen3-VL-4B: 500 unique images
   Qwen2.5-VL-3B: 500 unique images


In [3]:
# =============================================================================
# CELL C3: Combine All 3 Models Into One Dataset
# =============================================================================
combined_df = pd.concat(model_dfs, ignore_index=True)
combined_df.to_csv("combined_evaluation_dataset.csv", index=False)
print(f"✅ Combined: {len(combined_df)} rows across {combined_df['model'].nunique()} models")
files.download("combined_evaluation_dataset.csv")

✅ Combined: 1500 rows across 3 models


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
# =============================================================================
# CELL C4: Standard Captioning Evaluation Metrics (model-wise, real data)
# =============================================================================
!pip install -q bert_score rouge_score pycocoevalcap nltk

import re, warnings, time
warnings.filterwarnings("ignore")
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bertscore_score
from pycocoevalcap.cider.cider import Cider
from tqdm.auto import tqdm

bleu_smoothing = SmoothingFunction().method4
rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
cider_scorer = Cider()

def compute_bleu(candidate, references):
    cand_tokens = word_tokenize(candidate.lower())
    ref_tokens_list = [word_tokenize(r.lower()) for r in references]
    return sentence_bleu(ref_tokens_list, cand_tokens, smoothing_function=bleu_smoothing)

def compute_rouge(candidate, references):
    best_r1, best_rl = 0.0, 0.0
    for ref in references:
        s = rouge.score(ref, candidate)
        best_r1 = max(best_r1, s["rouge1"].fmeasure)
        best_rl = max(best_rl, s["rougeL"].fmeasure)
    return best_r1, best_rl

def clean_tokenize_sentence(text):
    text = re.sub(r"[^a-zA-Z0-9\s]", "", str(text).lower())
    return " ".join(text.split())

def build_cider_inputs(df, caption_col):
    gts, res = {}, {}
    for _, row in df.iterrows():
        img_key = f"{row['model']}_{row['image_id']}"
        refs = row["reference_captions"].split(" | ")
        gts[img_key] = [clean_tokenize_sentence(r) for r in refs]
        res[img_key] = [clean_tokenize_sentence(row[caption_col])]
    return gts, res

metric_rows = []
model_names = list(combined_df["model"].unique())
job_bar = tqdm(total=len(model_names) * 2, desc="📊 Overall metrics progress", unit="job")
t0 = time.time()

for model_name in model_names:
    model_df = combined_df[combined_df["model"] == model_name].reset_index(drop=True)

    for caption_type, caption_col in [("Baseline", "baseline_caption"), ("Grounded", "grounded_caption")]:
        candidates = model_df[caption_col].astype(str).tolist()
        references_list = [row.split(" | ") for row in model_df["reference_captions"].astype(str).tolist()]

        bleu_scores = [
            compute_bleu(c, r) for c, r in
            tqdm(zip(candidates, references_list), total=len(candidates),
                 desc=f"  BLEU [{model_name}/{caption_type}]", leave=False)
        ]
        avg_bleu = float(np.mean(bleu_scores))

        r1s, rls = [], []
        for c, r in tqdm(zip(candidates, references_list), total=len(candidates),
                          desc=f"  ROUGE [{model_name}/{caption_type}]", leave=False):
            r1, rl = compute_rouge(c, r)
            r1s.append(r1); rls.append(rl)
        avg_rouge1, avg_rougeL = float(np.mean(r1s)), float(np.mean(rls))

        P, R, F1 = bertscore_score(candidates, references_list, lang="en", verbose=True, rescale_with_baseline=True)
        avg_bertscore_f1 = float(F1.mean().item())

        gts, res = build_cider_inputs(model_df, caption_col)
        cider_score, _ = cider_scorer.compute_score(gts, res)

        metric_rows.append({
            "Model": model_name,
            "Caption Type": caption_type,
            "N Images": len(candidates),
            "BLEU-4": round(avg_bleu, 4),
            "ROUGE-1": round(avg_rouge1, 4),
            "ROUGE-L": round(avg_rougeL, 4),
            "BERTScore-F1": round(avg_bertscore_f1, 4),
            "CIDEr": round(float(cider_score), 4),
        })
        job_bar.update(1)

job_bar.close()
print(f"✅ All metrics computed in {(time.time()-t0)/60:.1f} minutes")

standard_metrics_df = pd.DataFrame(metric_rows)
standard_metrics_df.to_csv("standard_captioning_metrics_table.csv", index=False)
files.download("standard_captioning_metrics_table.csv")
print(standard_metrics_df.to_string(index=False))

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 9.1 MB/s eta 0:00:00


📊 Overall metrics progress:   0%|          | 0/6 [00:00<?, ?job/s]

  BLEU [LLaVA-1.5-7B/Baseline]:   0%|          | 0/500 [00:00<?, ?it/s]

  ROUGE [LLaVA-1.5-7B/Baseline]:   0%|          | 0/500 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

...

  0%|          | 0/40 [00:00<?, ?it/s]

done in 14.61 seconds, 171.27 sentences/sec
✅ All metrics computed in 2.2 minutes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

        Model Caption Type  N Images  BLEU-4  ROUGE-1  ROUGE-L  BERTScore-F1  CIDEr
 LLaVA-1.5-7B     Baseline       500  0.0647   0.2547   0.2186        0.3499 0.0003
 LLaVA-1.5-7B     Grounded       500  0.0711   0.2741   0.2367        0.3697 0.0015
  Qwen3-VL-4B     Baseline       500  0.0712   0.2706   0.2300        0.3551 0.0238
  Qwen3-VL-4B     Grounded       500  0.0500   0.2254   0.1903        0.2878 0.0001
Qwen2.5-VL-3B     Baseline       500  0.1883   0.4955   0.4416        0.5542 0.5833
Qwen2.5-VL-3B     Grounded       500  0.0575   0.2613   0.2173        0.3208 0.0008


In [5]:
# =============================================================================
# CELL C5: Final Research-Format Table (paper-ready) + Download
# =============================================================================
pivot_df = standard_metrics_df.pivot(
    index="Model", columns="Caption Type",
    values=["BLEU-4", "ROUGE-1", "ROUGE-L", "BERTScore-F1", "CIDEr"]
)
pivot_df.columns = [f"{metric} ({ctype})" for metric, ctype in pivot_df.columns]
pivot_df = pivot_df.reset_index()

print("\n" + "=" * 100)
print("📊 FINAL RESEARCH-FORMAT TABLE — Model-wise Evaluation (500 real COCO 2014 images)")
print("=" * 100)
print(pivot_df.to_string(index=False))

pivot_df.to_csv("final_research_table.csv", index=False)
with open("final_research_table.md", "w") as f:
    f.write(pivot_df.to_markdown(index=False))
with open("final_research_table.tex", "w") as f:
    f.write(pivot_df.to_latex(index=False, float_format="%.4f"))

for fname in ["final_research_table.csv", "final_research_table.md", "final_research_table.tex"]:
    files.download(fname)

print("\n🎉 Project complete — real 500 COCO images, 3 real models, 0 dummy data.")


📊 FINAL RESEARCH-FORMAT TABLE — Model-wise Evaluation (500 real COCO 2014 images)
        Model  BLEU-4 (Baseline)  BLEU-4 (Grounded)  ROUGE-1 (Baseline)  ROUGE-1 (Grounded)  ROUGE-L (Baseline)  ROUGE-L (Grounded)  BERTScore-F1 (Baseline)  BERTScore-F1 (Grounded)  CIDEr (Baseline)  CIDEr (Grounded)
 LLaVA-1.5-7B             0.0647             0.0711              0.2547              0.2741              0.2186              0.2367                   0.3499                   0.3697            0.0003            0.0015
Qwen2.5-VL-3B             0.1883             0.0575              0.4955              0.2613              0.4416              0.2173                   0.5542                   0.3208            0.5833            0.0008
  Qwen3-VL-4B             0.0712             0.0500              0.2706              0.2254              0.2300              0.1903                   0.3551                   0.2878            0.0238            0.0001


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 Project complete — real 500 COCO images, 3 real models, 0 dummy data.


In [6]:
# =============================================================================
# CELL C6: Final Model-Wise Table with Caption Change (Baseline → Grounded)
# =============================================================================
# WHAT: Builds the properly formatted table your professor wants — one row
# per model, each metric shown for Baseline and Grounded, PLUS a Δ (delta)
# column quantifying how much the caption changed once grounding was applied.
# This uses ONLY the real standard_metrics_df computed in Cell C4 — no new
# data, just a clean re-presentation with the deltas your professor needs.
# =============================================================================

# ---- Rebuild delta table from standard_metrics_df (already computed) ----
delta_rows = []
for model_name in standard_metrics_df["Model"].unique():
    base_row = standard_metrics_df[
        (standard_metrics_df["Model"] == model_name) &
        (standard_metrics_df["Caption Type"] == "Baseline")
    ].iloc[0]
    ground_row = standard_metrics_df[
        (standard_metrics_df["Model"] == model_name) &
        (standard_metrics_df["Caption Type"] == "Grounded")
    ].iloc[0]

    delta_rows.append({
        "Model": model_name,

        "Δ BLEU-4": round(ground_row["BLEU-4"] - base_row["BLEU-4"], 4),

        "Δ ROUGE-1": round(ground_row["ROUGE-1"] - base_row["ROUGE-1"], 4),

        "Δ ROUGE-L": round(ground_row["ROUGE-L"] - base_row["ROUGE-L"], 4),

        "Δ BERTScore": round(ground_row["BERTScore-F1"] - base_row["BERTScore-F1"], 4),

        "Δ CIDEr": round(ground_row["CIDEr"] - base_row["CIDEr"], 4),
    })

delta_df = pd.DataFrame(delta_rows)

print("\n" + "=" * 100)
print("📊 FINAL MODEL-WISE TABLE — Caption Change from Baseline → Grounded (500 real COCO images)")
print("=" * 100)
print(delta_df.to_string(index=False))

# ---- Save in 3 formats for your report ----
delta_df.to_csv("final_caption_change_table.csv", index=False)
with open("final_caption_change_table.md", "w") as f:
    f.write(delta_df.to_markdown(index=False))
with open("final_caption_change_table.tex", "w") as f:
    f.write(delta_df.to_latex(index=False, float_format="%.4f"))

for fname in ["final_caption_change_table.csv", "final_caption_change_table.md", "final_caption_change_table.tex"]:
    files.download(fname)

print("\n✅ Saved and downloaded: CSV / Markdown / LaTeX versions of the caption-change table.")


📊 FINAL MODEL-WISE TABLE — Caption Change from Baseline → Grounded (500 real COCO images)
        Model  Δ BLEU-4  Δ ROUGE-1  Δ ROUGE-L  Δ BERTScore  Δ CIDEr
 LLaVA-1.5-7B    0.0064     0.0194     0.0181       0.0198   0.0012
  Qwen3-VL-4B   -0.0212    -0.0452    -0.0397      -0.0673  -0.0237
Qwen2.5-VL-3B   -0.1308    -0.2342    -0.2243      -0.2334  -0.5825


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Saved and downloaded: CSV / Markdown / LaTeX versions of the caption-change table.


In [7]:
# =============================================================================
# CELL C7: Manual Sanity Check — Inspect Qwen2.5-VL-3B Baseline Captions
# =============================================================================
# WHY: Qwen2.5-VL-3B's baseline scores are dramatically higher than every
# other model/prompt combination (CIDEr 0.58 vs <0.03 elsewhere). Before
# reporting this to your professor, verify it's a real result and not a
# data artifact (e.g. baseline caption accidentally duplicating a reference).
# =============================================================================

qwen_baseline_df = combined_df[combined_df["model"] == "Qwen2.5-VL-3B"].head(8)

for _, row in qwen_baseline_df.iterrows():
    print(f"\n🔹 Image ID: {row['image_id']}")
    print(f"  • Baseline Caption : {row['baseline_caption']}")
    print(f"  • Reference (1st)  : {row['reference_captions'].split(' | ')[0]}")
    print("  " + "-" * 70)


🔹 Image ID: 105156
  • Baseline Caption : A white horse is harnessed to a cart with colorful blankets, standing on a street next to a sign with Chinese characters. The background features trees and mountains under a clear sky.
  • Reference (1st)  : A horse stands still attached to a passenger coach. 
  ----------------------------------------------------------------------

🔹 Image ID: 22861
  • Baseline Caption : A pepperoni pizza in a cast iron pan with a spatula.
  • Reference (1st)  : A fresh pepperoni pizza seems to have a couple of slices missing. 
  ----------------------------------------------------------------------

🔹 Image ID: 258529
  • Baseline Caption : A small cabin sits in the woods with a stone path leading up to it.
  • Reference (1st)  : A pathway leading to an outdoor open shed.
  ----------------------------------------------------------------------

🔹 Image ID: 229840
  • Baseline Caption : A dog is sniffing the leg of a horse through a wire fence.
  • Reference

In [8]:
# =============================================================================
# CELL C8: Check the Per-Image CIDEr Distribution for Qwen2.5-VL-3B Baseline
# =============================================================================
# WHY: A high average CIDEr can come from consistently good captions, OR from
# a few extreme outliers dragging the average up. This tells you which one
# you're actually looking at — important for how you describe this result.
# =============================================================================
import numpy as np

qwen_base_df = combined_df[combined_df["model"] == "Qwen2.5-VL-3B"].reset_index(drop=True)

gts, res = build_cider_inputs(qwen_base_df, "baseline_caption")
_, per_image_scores = cider_scorer.compute_score(gts, res)

per_image_scores = np.array(per_image_scores)

print(f"Mean CIDEr:   {per_image_scores.mean():.4f}")
print(f"Median CIDEr: {np.median(per_image_scores):.4f}")
print(f"Max CIDEr:    {per_image_scores.max():.4f}")
print(f"Min CIDEr:    {per_image_scores.min():.4f}")
print(f"Std Dev:      {per_image_scores.std():.4f}")
print(f"\n% of images scoring above 1.0: {(per_image_scores > 1.0).mean()*100:.1f}%")
print(f"% of images scoring above 2.0: {(per_image_scores > 2.0).mean()*100:.1f}%")
print(f"% of images scoring 0.0:       {(per_image_scores == 0).mean()*100:.1f}%")

Mean CIDEr:   0.5833
Median CIDEr: 0.3511
Max CIDEr:    3.6057
Min CIDEr:    0.0000
Std Dev:      0.6762

% of images scoring above 1.0: 22.2%
% of images scoring above 2.0: 4.8%
% of images scoring 0.0:       0.0%


In [9]:
# =============================================================================
# CELL C9: Caption Change Table — Baseline vs Grounded (Direct Comparison)
# =============================================================================
# WHAT: For each image, treats the BASELINE caption as the "reference" and
# the GROUNDED caption as the "candidate", and computes BLEU-4, ROUGE-1,
# ROUGE-L, BERTScore, and CIDEr directly BETWEEN the two captions.
# This tells you how much the caption text itself changed once grounding
# was applied — a LOW score means the grounded caption is very different
# from the baseline (big change); a HIGH score means grounding barely
# altered the wording (small change).
# This is a separate, more direct measurement than the delta table in
# Cell C6, which instead compared each caption type against the human
# ground-truth references and only showed the DIFFERENCE in those scores.
# =============================================================================

change_rows = []
job_bar2 = tqdm(total=len(combined_df["model"].unique()), desc="📊 Computing caption-change metrics", unit="model")

for model_name in combined_df["model"].unique():
    model_df = combined_df[combined_df["model"] == model_name].reset_index(drop=True)

    baseline_caps = model_df["baseline_caption"].astype(str).tolist()
    grounded_caps = model_df["grounded_caption"].astype(str).tolist()

    # ---- BLEU-4: grounded caption scored against baseline as its "reference" ----
    bleu_scores = [
        compute_bleu(g, [b]) for g, b in zip(grounded_caps, baseline_caps)
    ]
    avg_bleu = float(np.mean(bleu_scores))

    # ---- ROUGE-1 / ROUGE-L: grounded vs baseline directly ----
    r1s, rls = [], []
    for g, b in zip(grounded_caps, baseline_caps):
        r1, rl = compute_rouge(g, [b])
        r1s.append(r1); rls.append(rl)
    avg_rouge1, avg_rougeL = float(np.mean(r1s)), float(np.mean(rls))

    # ---- BERTScore: grounded vs baseline directly ----
    P, R, F1 = bertscore_score(
        grounded_caps, [[b] for b in baseline_caps],
        lang="en", verbose=False, rescale_with_baseline=True,
    )
    avg_bertscore_f1 = float(F1.mean().item())

    # ---- CIDEr: grounded vs baseline directly ----
    gts_change, res_change = {}, {}
    for idx, (b, g) in enumerate(zip(baseline_caps, grounded_caps)):
        key = f"{model_name}_{idx}"
        gts_change[key] = [clean_tokenize_sentence(b)]
        res_change[key] = [clean_tokenize_sentence(g)]
    cider_score, _ = cider_scorer.compute_score(gts_change, res_change)

    change_rows.append({
        "Model": model_name,
        "BLEU-4 (Base↔Grounded)": round(avg_bleu, 4),
        "ROUGE-1 (Base↔Grounded)": round(avg_rouge1, 4),
        "ROUGE-L (Base↔Grounded)": round(avg_rougeL, 4),
        "BERTScore (Base↔Grounded)": round(avg_bertscore_f1, 4),
        "CIDEr (Base↔Grounded)": round(float(cider_score), 4),
        "BERTScore Dissimilarity (%)": round((1 - avg_bertscore_f1) * 100, 2),
    })
    job_bar2.update(1)

job_bar2.close()

caption_change_df = pd.DataFrame(change_rows)

print("\n" + "=" * 100)
print("📊 CAPTION CHANGE TABLE — Baseline vs Grounded Caption Similarity (500 real COCO images)")
print("=" * 100)
print("Interpretation: LOW scores = grounded caption is very different from baseline (big change)")
print("                HIGH scores = grounding barely changed the wording (small change)")
print("=" * 100)
print(caption_change_df.to_string(index=False))

# ---- Save in 3 formats for your report ----
caption_change_df.to_csv("caption_change_table.csv", index=False)
with open("caption_change_table.md", "w") as f:
    f.write(caption_change_df.to_markdown(index=False))
with open("caption_change_table.tex", "w") as f:
    f.write(caption_change_df.to_latex(index=False, float_format="%.4f"))

for fname in ["caption_change_table.csv", "caption_change_table.md", "caption_change_table.tex"]:
    files.download(fname)

print("\n✅ Saved and downloaded: CSV / Markdown / LaTeX versions of the caption-change table.")

📊 Computing caption-change metrics:   0%|          | 0/3 [00:00<?, ?model/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Saved and downloaded: CSV / Markdown / LaTeX versions of the caption-change table.
